In [ ]:
import numpy as np
import pandas as pd
import os
import time
import json 
import warnings
from scipy.stats import t as t_dist
from arch import arch_model
import yfinance as yf

warnings.filterwarnings("ignore")
MASTER_SEED = 42

def _generate_garch_data():
    np.random.seed(MASTER_SEED)
    data = yf.download("^FTSE", start="2019-01-01", end="2025-01-01", progress=False)
    close = data['Close']['^FTSE'] if isinstance(data.columns, pd.MultiIndex) else data['Close']
    S0 = float(close.iloc[-1])
    ret = 100 * np.log(close / close.shift(1)).dropna()
    res = arch_model(ret, vol='Garch', p=1, q=1, dist='t').fit(disp='off')
    omega, alpha, beta, nu = res.params['omega'], res.params['alpha[1]'], res.params['beta[1]'], res.params['nu']

    n_sims, T_mat, dpy, mu_adj = 100_000, 5, 252, 0.04
    n_days = T_mat * dpy
    np.random.seed(MASTER_SEED)
    K_assign = S0 * np.random.choice(np.arange(0.6, 1.6, 0.1), size=n_sims)

    np.random.seed(MASTER_SEED + 5000)
    var0 = omega / (1 - alpha - beta)
    S_paths = np.zeros((n_sims, n_days)); sigma_paths = np.zeros((n_sims, n_days))
    S_paths[:,0] = S0; sigma_paths[:,0] = np.sqrt(var0)
    var_cur, S_cur = np.full(n_sims, var0), np.full(n_sims, S0)
    for d in range(1, n_days):
        z = t_dist.rvs(df=nu, size=n_sims) / np.sqrt(nu/(nu-2))
        sig = np.sqrt(var_cur); eps = sig * z
        S_cur = S_cur * np.exp((mu_adj + eps) / 100)
        S_paths[:,d] = S_cur; sigma_paths[:,d] = sig
        var_cur = np.minimum(omega + alpha*(eps**2) + beta*var_cur, 25.0)

    df = pd.DataFrame({
        'simulation': np.repeat(np.arange(n_sims), n_days),
        'day': np.tile(np.arange(n_days), n_sims),
        'S': S_paths.flatten(),
        'K': np.repeat(K_assign, n_days),
        'T': np.tile(T_mat - np.arange(n_days)/dpy, n_sims),
        'sigma': sigma_paths.flatten() * np.sqrt(dpy) / 100
    }).sort_values(['simulation','day']).reset_index(drop=True)
    return df

df = _generate_garch_data()
    

print(f"Data ready: {len(df):,} rows, {df['simulation'].nunique():,} simulations")

In [ ]:
from scipy.stats import norm

def _bs_call(S, K, T, r, sigma):
    mask = T > 1e-10
    price = np.maximum(S - K, 0)
    if np.any(mask):
        Sv, Kv, Tv, sv = (np.where(mask, x, d) for x, d in [(S,1),(K,1),(T,1),(sigma,.2)])
        d1 = (np.log(Sv/Kv) + (r + .5*sv**2)*Tv) / (sv*np.sqrt(Tv))
        price = np.where(mask, Sv*norm.cdf(d1) - Kv*np.exp(-r*Tv)*norm.cdf(d1 - sv*np.sqrt(Tv)), price)
    return price

CSV = 'ftse_garch_simulation_data_T5.csv'
df = pd.read_csv(CSV) if os.path.exists(CSV) else _generate_garch_data()

# Add risk-free rate and Black-Scholes call price
df['r'] = 0.05
if 'call_price' not in df.columns:
    df['call_price'] = _bs_call(df['S'].values, df['K'].values, df['T'].values, 0.05, df['sigma'].values)

print(f"Data: {len(df):,} rows | {df['simulation'].nunique():,} sims | call_price mean={df['call_price'].mean():.1f}")
print(df.head())